# Stage 4: Evaluate Text Variables

**Purpose**: Resolves formula-based text variables by substituting parameter values and evaluating expressions.

**Input**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage3.json`  
**Output**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage4.json`

## What this notebook does

BC3 text variables can contain:

- **Parameter placeholders**: `%A`, `$B` → replaced with actual parameter values
- **Conditional expressions**: `(%A=B)` → evaluates to 1 or 0
- **Array indexing**: `$L(a,%C)` → looks up values in multi-dimensional arrays
- **Mathematical formulas**: Translated to Python and evaluated

### Example transformation

```
Original:  "$L(%A,%B)" where L is a lookup table and A="a", B="c"
Resolved:  "Concrete type HE-20"
```

The notebook processes data in chunks (1000 items) to handle memory constraints with large datasets. Text variables are processed in two passes:
1. Direct parameter replacements
2. Resolution of inter-variable dependencies

In [12]:
!pip install ijson

In [13]:
import os
import json
import ijson
from typing import Iterator, Dict, Any
import re
import traceback
from math import sin, cos, tan, asin, acos, atan, atan2, sqrt, fabs as ABS, floor as INT
import time

def is_already_quoted(text):
    """Check if a string is already enclosed in double quotes.
    
    Args:
        text (str): Text to check for quotes
        
    Returns:
        bool: True if text starts and ends with double quotes
    """
    return bool(re.match(r'^".*"$', text.strip()))

def safe_quote(text):
    """Add double quotes to a string if not already quoted.
    
    Args:
        text (str): Text to quote
        
    Returns:
        str: Text with surrounding double quotes
    """
    if is_already_quoted(text):
        return text
    return f'"{text}"'

def is_formula(text):
    """Determine if text contains formula operators or functions.
    
    Args:
        text: String to check for formula indicators
        
    Returns:
        bool: True if text contains formula elements
    """
    if not isinstance(text, str):
        return False
    formula_indicators = ['+', '-', '*', '/', '=', '<>', '@', '&', '^', 
                         'ABS', 'INT', 'ROUND', 'SIN', 'COS', 'TAN', 
                         'ASIN', 'ACOS', 'ATAN', 'ATAN2', 'SQRT', 'ATOF', 'FTOA',
                         '==']
    return any(indicator in text for indicator in formula_indicators)

def is_comparison_placeholder(text, placeholder_type, key):
    """Check if a placeholder is used in a comparison operation.
    
    Args:
        text (str): Text containing placeholder
        placeholder_type (str): Type of placeholder (% or $)
        key (str): Placeholder key
        
    Returns:
        bool: True if placeholder is used in comparison
    """
    pattern = rf'[{placeholder_type}]{key}\s*==|==\s*[{placeholder_type}]{key}'
    return bool(re.search(pattern, text))

def find_text_variable_references(text):
    """Find references to other text variables ($VAR or $VAR(args)).
    
    Args:
        text (str): Text to search for references
        
    Returns:
        list: List of referenced variable names
    """
    if not isinstance(text, str):
        return []
    pattern = r'\$([A-Za-z0-9]+)(?:\(.*?\))?'
    return re.findall(pattern, text)

def find_parameter_placeholders(text):
    """Find parameter placeholders not part of text variable references.
    
    Args:
        text (str): Text to search for placeholders
        
    Returns:
        list: List of parameter placeholder names
    """
    if not isinstance(text, str):
        return []
    # First find all $VAR(args) patterns and temporarily remove them
    text_var_pattern = r'\$([A-Za-z0-9]+)(?:\(.*?\))'
    temp_text = re.sub(text_var_pattern, '', text)
    
    # Now find remaining % or $ prefixed parameters
    return re.findall(r'[%$]([A-Za-z0-9]+)', temp_text)

def replace_parameter_placeholder(text, parameters, placeholder_type, key, is_formula_var=False):
    """Replace a parameter placeholder with its value or label.
    
    Args:
        text (str): Text containing placeholder
        parameters (dict): Parameter definitions
        placeholder_type (str): Type of placeholder (% or $)
        key (str): Parameter key
        is_formula_var (bool): Whether replacement is in a formula
        
    Returns:
        str: Text with placeholder replaced
    """
    try:
        if not parameters.get(key):
            return text
        
        param_value = parameters[key]['values'][0]
        is_comparison = is_formula_var and is_comparison_placeholder(text, placeholder_type, key)
        replacement = param_value['label'] if is_comparison else param_value['value']
        
        # Only quote if it's a comparison or a string in a formula
        if is_formula_var and not is_already_quoted(replacement):
            replacement = safe_quote(replacement)
        
        pattern = f'[{placeholder_type}]{key}\\b'
        return re.sub(pattern, replacement, text)
    except Exception as e:
        debug_print("replace_parameter_placeholder", text, e)
        return text

def resolve_text_variable_reference(reference, text_vars):
    """Resolve a reference to another text variable.
    
    Args:
        reference (str): Reference string ($VAR or $VAR(args))
        text_vars (dict): Dictionary of text variables
        
    Returns:
        str: Resolved value of the reference
    """
    try:
        match = re.match(r'\$(\w+)(?:\((.*?)\))?', reference)
        if not match:
            return reference
            
        var_name, args = match.groups()
        if var_name not in text_vars:
            return reference
            
        referenced_var = text_vars[var_name]
        resolved_value = referenced_var.get('evaluated', '')
        
        if isinstance(resolved_value, str):
            if not is_already_quoted(resolved_value):
                return safe_quote(resolved_value)
            return resolved_value
            
        if args and isinstance(resolved_value, list):
            arg_list = [arg.strip() for arg in args.split(',')]
            result = resolved_value
            for arg in arg_list:
                if isinstance(result, list):
                    try:
                        index = int(arg) if arg.isdigit() else 0
                        result = result[index]
                    except (IndexError, ValueError):
                        return reference
            if isinstance(result, str) and not is_already_quoted(result):
                return safe_quote(result)
            return result
            
        return reference
    except Exception as e:
        #print(f"Error resolving text variable reference {reference}: {str(e)}")
        return reference

def evaluate_formula(formula_text):
    """Evaluate a formula string as Python code.
    
    Args:
        formula_text (str): Formula to evaluate
        
    Returns:
        str: Result of formula evaluation
    """
    try:
        
        # Clean up the formula - remove any double quotes next to each other
        clean_formula = re.sub(r'"\s*"', '" "', formula_text)        
        python_formula = translate_formula_to_python(clean_formula)        
        result = eval(python_formula)
        return result
    except Exception as e:
        debug_print("evaluate_formula", formula_text, e)
        return formula_text

def process_text_variables(data):
    """First pass: Process text variables with direct parameter replacements.
    
    Args:
        data (dict): Input JSON data
        
    Returns:
        dict: Processed data with text variables updated
    """
    try:
        for key, item in data.items():
            if 'text_variables' not in item or 'parameters' not in item:
                continue
                
            text_vars = item['text_variables']
            parameters = item['parameters']
            processed_vars = {}
            
            for var_key, var_value in text_vars.items():
                try:
                    is_formula_var = is_formula(str(var_value))
                    
                    if isinstance(var_value, str):
                        processed = process_single_item(var_value, parameters, is_formula_var)
                        processed_vars[var_key] = {
                            'original': processed['original'],
                            'replaced': processed['replaced'],
                            'evaluated': processed['evaluated'],
                            'is_formula': is_formula_var,
                            'dependencies': processed['dependencies'],
                            'fully_processed': not processed['dependencies'] and 
                                            not bool(find_parameter_placeholders(processed['replaced']))
                        }
                    elif isinstance(var_value, list):
                        # Handle list of items
                        processed_list = []
                        all_dependencies = set()
                        
                        for list_item in var_value:
                            if isinstance(list_item, list):
                                # Handle nested list
                                nested_processed = [
                                    process_single_item(x, parameters, is_formula_var)
                                    for x in list_item
                                ]
                                processed_list.append(nested_processed)
                                for x in nested_processed:
                                    all_dependencies.update(x['dependencies'])
                            else:
                                # Handle single item
                                item_processed = process_single_item(list_item, parameters, is_formula_var)
                                processed_list.append(item_processed)
                                all_dependencies.update(item_processed['dependencies'])
                        
                        # Create the processed variable entry for lists
                        processed_vars[var_key] = {
                            'original': var_value,
                            'replaced': [
                                [x['replaced'] for x in item] if isinstance(item, list)
                                else item['replaced'] for item in processed_list
                            ],
                            'evaluated': [
                                [x['evaluated'] for x in item] if isinstance(item, list)
                                else item['evaluated'] for item in processed_list
                            ],
                            'is_formula': is_formula_var,
                            'dependencies': list(all_dependencies),
                            'fully_processed': not all_dependencies and 
                                            not any(find_parameter_placeholders(str(x['replaced'])) 
                                                for item in processed_list
                                                for x in (item if isinstance(item, list) else [item]))
                        }
                except Exception as e:
                    debug_print(f"Processing var {var_key}", var_value, e)
                    processed_vars[var_key] = {
                        'original': var_value,
                        'replaced': var_value,
                        'evaluated': var_value,
                        'is_formula': False,
                        'dependencies': [],
                        'fully_processed': False,
                        'error': str(e)
                    }
            
            item['text_variables'] = processed_vars
        
        return data
    except Exception as e:
        debug_print("process_text_variables", "main process", e)
        raise

def process_unresolved_variables(data):
    """Second pass: Process variables that depend on other text variables.
    
    Args:
        data (dict): Data from first pass
        
    Returns:
        dict: Final processed data with all variables resolved
    """
    for key, item in data.items():
        if 'text_variables' not in item:
            continue
            
        text_vars = item['text_variables']
        parameters = item.get('parameters', {})
        
        # Find unprocessed variables
        unprocessed_vars = {
            var_key: var_data 
            for var_key, var_data in text_vars.items() 
            if not var_data.get('fully_processed', False)
        }
        
        for var_key, var_data in unprocessed_vars.items():
            try:
                original = var_data['original']
                is_formula_var = var_data['is_formula']
                replaced_text = original

                # Process text variable references first
                references = find_text_variable_references(original)
                for ref in references:
                    ref_pattern = rf'\${ref}(?:\([^)]*\))?'
                    matches = list(re.finditer(ref_pattern, replaced_text))
                    
                    for match in reversed(matches):  # Process from end to start to avoid position issues
                        full_ref = match.group(0)
                        resolved_value = resolve_text_variable_reference(full_ref, text_vars)
                        # No need for extra quotes as resolve_text_variable_reference handles quoting
                        replaced_text = replaced_text[:match.start()] + str(resolved_value) + replaced_text[match.end():]

                # Then process parameter placeholders
                placeholders = find_parameter_placeholders(replaced_text)
                for placeholder in placeholders:
                    for prefix in ['%', '$']:
                        replaced_text = replace_parameter_placeholder(
                            replaced_text, parameters, prefix, placeholder,
                            is_formula_var=is_formula_var
                        )

                # Clean up any potential double spaces or unnecessary quotes
                # but be careful not to modify quoted strings
                replaced_text = re.sub(r'\s+', ' ', replaced_text)

                # Update the variable
                text_vars[var_key].update({
                    'replaced': replaced_text,
                    'evaluated': evaluate_formula(replaced_text) if is_formula_var else replaced_text,
                    'fully_processed': True
                })
                
            except Exception as e:
                debug_print(f"Error processing {var_key}", str(e))
                text_vars[var_key].update({
                    'error': str(e),
                    'fully_processed': False
                })
    
    return data

def translate_formula_to_python(formula):
    """
    Translate formula string into Python syntax.
    Handles operators carefully to avoid double/triple equals issues.
    """
    # Dictionary to hold the Python equivalents of custom operators and functions
    python_equivalents = {
        "@": " or ",
        "&": " and ",
        "^": "**",
        "<>": "!=",
        "ABS": "ABS",
        "INT": "INT",
        "ROUND": "round",
        "SIN": "sin",
        "COS": "cos",
        "TAN": "tan",
        "ASIN": "asin",
        "ACOS": "acos",
        "ATAN": "atan",
        "ATAN2": "atan2",
        "SQRT": "sqrt",
        "ATOF": "float",
        "FTOA": "str",
    }

    # Replace operators and function names, but protect == from being modified
    for custom, py_equiv in python_equivalents.items():
        # Skip the = to == replacement as we handle == separately
        formula = re.sub(r'\b' + re.escape(custom) + r'\b', py_equiv, formula)
    
    # Transform conditions into Python's ternary condition syntax
    formula = re.sub(r'\(%(\w)=(\w)\)', r'(1 if \1=="\2" else 0)', formula)
   
    return formula

# Function to quote single characters (parameter labels)
def quote_second_term(text):
    # This regular expression matches an operator followed by a variable or sequence of characters
    pattern = r'(\w+\s*([=<>!]=|[=<>])\s*)(\w+)'

    # This function is used to replace the matched pattern
    def replacer(match):
        return f'{match.group(1)}"{match.group(3)}"'

    # Replace all occurrences in the text using the pattern and replacer function
    return re.sub(pattern, replacer, text)

def debug_print(context, value, error=None):
    """Print debug information with context"""
    #print(f"\nDEBUG [{context}]:")
    #print(f"Value: {value}")
    if error:
        pass
        #print(f"Error: {str(error)}")
        #print(f"Type: {type(value)}")
        #print(f"Traceback: {traceback.format_exc()}")

def process_single_item(text, parameters, is_formula_var):
    """Process a single text item."""
    try:
        if not isinstance(text, str):
            return {
                'original': text,
                'replaced': text,
                'evaluated': text,
                'dependencies': []
            }

        # Check for dependencies first
        dependencies = find_text_variable_references(text)
        if dependencies:
            return {
                'original': text,
                'replaced': text,
                'evaluated': text,
                'dependencies': dependencies
            }

        # Process parameter placeholders
        replaced_text = text
        placeholders = find_parameter_placeholders(text)
        
        for placeholder in placeholders:
            use_label = is_formula_var
            for prefix in ['%', '$']:
                new_text = replace_parameter_placeholder(
                    replaced_text, parameters, prefix, placeholder, use_label
                )
                replaced_text = new_text

        # Evaluate if it's a formula
        evaluated_text = evaluate_formula(replaced_text) if is_formula_var else replaced_text

        return {
            'original': text,
            'replaced': replaced_text,
            'evaluated': evaluated_text,
            'dependencies': []
        }
    except Exception as e:
        debug_print("process_single_item", text, e)
        return {
            'original': text,
            'replaced': text,
            'evaluated': text,
            'dependencies': []
        }

# Generate a structured filename
# Input: Source file path, stage number, output directory, and file extension.
# Output: A structured file name for the output file.
def generate_filename(input_file, stage, output_dir, extension="json"):
    """Generate a structured filename with stage and timestamp."""
    base_name = os.path.splitext(os.path.basename(input_file))[0]  # Get base name of the source file
    file_name = f"{base_name}_stage{stage}.{extension}"
    return os.path.join(output_dir, file_name)

# Create output directories dynamically within the source file's path
# Input: Source file path.
# Output: Path to the created output directory.

def create_output_dirs(input_file):
    """Create and return the output directory path within the source file's directory."""
    input_dir = os.path.dirname(input_file)
    output_dir = input_dir # Same directory
    os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists
    return output_dir

CHUNK_SIZE = 1000  # Adjust based on available memory

def read_json_in_chunks(filename: str, chunk_size: int) -> Iterator[Dict[str, Any]]:
    """Read large JSON file in chunks using manual file reading."""
    chunk = {}
    items_processed = 0
    
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
        total_items = len(data)
        print(f"Total items to process: {total_items}")
        
        for key, value in data.items():
            chunk[key] = value
            items_processed += 1
            
            if len(chunk) >= chunk_size:
                print(f"Yielding chunk of {len(chunk)} items. Progress: {items_processed}/{total_items}")
                yield chunk
                chunk = {}
        
        if chunk:
            print(f"Yielding final chunk of {len(chunk)} items. Total processed: {items_processed}")
            yield chunk

def process_chunk(chunk: Dict[str, Any], output_dir: str, chunk_num: int) -> str:
    """Process a single chunk of data."""
    print(f"Processing chunk {chunk_num} with {len(chunk)} items")
    transformed_data = process_text_variables(chunk)
    final_data = process_unresolved_variables(transformed_data)
    
    chunk_file = os.path.join(output_dir, f'chunk_{chunk_num}.json')
    with open(chunk_file, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, ensure_ascii=False, indent=4)
    
    return chunk_file

def merge_chunks(chunk_files: list, output_file: str):
    """Merge processed chunks into final output."""
    print(f"Merging {len(chunk_files)} chunks")
    merged_data = {}
    
    for chunk_file in chunk_files:
        with open(chunk_file, 'r', encoding='utf-8') as f:
            chunk_data = json.load(f)
            merged_data.update(chunk_data)
        os.remove(chunk_file)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(merged_data, f, ensure_ascii=False, indent=4)
    print(f"Merged data contains {len(merged_data)} items")

def main(input_file: str):
    """Process large JSON file in chunks."""
    start_time = time.time()
    stage3_input = generate_filename(input_file, stage=3, output_dir=os.path.dirname(input_file))
    output_dir = create_output_dirs(input_file)
    chunk_files = []
    
    try:
        print(f"Reading from: {stage3_input}")
        for i, chunk in enumerate(read_json_in_chunks(stage3_input, CHUNK_SIZE), 1):
            chunk_file = process_chunk(chunk, output_dir, i)
            chunk_files.append(chunk_file)
            
        stage4_output = generate_filename(input_file, stage=4, output_dir=output_dir)
        merge_chunks(chunk_files, stage4_output)
        
        print(f"Processing completed in {time.time() - start_time:.2f} seconds")
        print(f"Output saved to: {stage4_output}")
        
    except Exception as e:
        print(f"Error during processing:")
        traceback.print_exc()

# Input Format:
# {
#     "item_key": {
#         "text_variables": {
#             "var_name": string or list,  # Original text with placeholders
#             ...
#         },
#         "parameters": {
#             "param_name": {
#                 "label": string,
#                 "values": [
#                     {"label": string, "value": string},
#                     ...
#                 ]
#             },
#             ...
#         }
#     },
#     ...
# }

# Output Format:
# {
#     "item_key": {
#         "text_variables": {
#             "var_name": {
#                 "original": string or list,  # Original text
#                 "replaced": string or list,  # Text with placeholders replaced
#                 "evaluated": string or list, # Final evaluated text
#                 "is_formula": boolean,       # Whether it contains formula operations
#                 "dependencies": list,        # Other text variables referenced
#                 "fully_processed": boolean   # Whether all replacements are complete
#             },
#             ...
#         },
#         "parameters": { ... }  # Original parameters structure
#     },
#     ...
# }

In [14]:
from utils import config

input_file = config.chapter_path("OBRA CIVIL")
main(input_file)

Reading from: /work/data/intermediate/OBRA CIVIL/OBRA CIVIL_stage3.json
Total items to process: 126938
Yielding chunk of 1000 items. Progress: 1000/126938
Processing chunk 1 with 1000 items
Yielding chunk of 1000 items. Progress: 2000/126938
Processing chunk 2 with 1000 items
Yielding chunk of 1000 items. Progress: 3000/126938
Processing chunk 3 with 1000 items
Yielding chunk of 1000 items. Progress: 4000/126938
Processing chunk 4 with 1000 items
Yielding chunk of 1000 items. Progress: 5000/126938
Processing chunk 5 with 1000 items
Yielding chunk of 1000 items. Progress: 6000/126938
Processing chunk 6 with 1000 items
Yielding chunk of 1000 items. Progress: 7000/126938
Processing chunk 7 with 1000 items
Yielding chunk of 1000 items. Progress: 8000/126938
Processing chunk 8 with 1000 items
Yielding chunk of 1000 items. Progress: 9000/126938
Processing chunk 9 with 1000 items
Yielding chunk of 1000 items. Progress: 10000/126938
Processing chunk 10 with 1000 items
Yielding chunk of 1000 it